# Preliminary experiment: is the main study worth running?

**Question.** Is there forgetting to prevent when a *pretrained* model learns a stream of domains, and does the unbalanced-WFR method (**U**) prevent more of it than its controls, at an affordable cost?

This notebook implements section "Preliminary experiment" of the research plan. It uses only the **development stream**, so the main study's test domains stay untouched. Hyperparameters may be adjusted on these results; the main study's may not.

| Condition | What it is |
|---|---|
| **LoRA-match** | one LoRA on every linear layer, with the same number of trainable parameters as U at its maximum of 8 components |
| **BAL** | the same model and algorithm as U, with normalised masses and fully plastic shared weights: the normalisation-only control |
| **U** | unbalanced WFR: evidence masses, stiffness min(1, A·b_c/m_c) and min(1, A·b/N), creation of components for uncovered data |
| **MoLoRA-router** | U's adapters, all present from the start, mixed per token by a learned top-2 router (plain gradient training) |

**Model:** Pythia-70M-deduped (frozen) + adapters. **Stream:** Number theory → Strongly correlated electrons → Computer vision → Quantitative finance → Sports news, 200 steps each, batch 16 × 512 tokens, never announced to the model. Seed 1 (shuffled order) is run for U and LoRA-match only.

**Runtime:** about 1.5 hours on a T4: data download about 10 minutes, the six runs about 50 minutes, the Pythia-160M timing probe about 10 minutes. Every finished run is saved to `My Drive/uwfr_prelim/`, and reruns skip it, so a disconnect loses at most one run.

**Run order:** all cells from top to bottom. The last cell prints the go/no-go table and the decision.

In [ ]:
!pip install -q "datasets>=2.18" "transformers>=4.40" scikit-learn

## 1. Setup and configuration
All settings are in `CFG`. U's settings are scaled to 200-step segments (the main study uses 300).

In [ ]:
# Preliminary experiment: unbalanced WFR (U) vs. controls on a pretrained Pythia model, development stream only.
import os, math, time, json, re, html, random, urllib.request, urllib.parse, csv, hashlib, glob, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OFFLINE = os.environ.get("UWFR_OFFLINE") == "1"      # code check without internet: tiny random model, byte tokenizer
USE_DRIVE = not OFFLINE and not os.environ.get("UWFR_OUT")
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/uwfr_prelim"
else:
    ROOT = os.environ.get("UWFR_OUT", "uwfr_prelim")
os.makedirs(ROOT, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| outputs:", ROOT)

CFG = dict(
    model="EleutherAI/pythia-70m-deduped",
    seq_len=512, batch=16, steps_per_domain=200, lr=5e-4, warmup=50, clip=1.0, amp=True,
    # adapters
    shared_rank=32, comp_rank=16, comp_layers=2, K_max=8,
    # unbalanced WFR (settings scaled to 200-step segments; tuned on this development stream only)
    prefix=128,        # E-step on the first 128 tokens of each sequence
    A_p=130, A_s=130,  # M-step horizon (steps): factors min(1, A b_c/m_c) and min(1, A b/N_fit)
    delta=0.10,        # covered if loss_c(x) <= level_c + delta   (nats per token)
    level_q=0.75, level_rate=0.05,
    nu_region=0.3,     # creation needs > 30% of a batch uncovered by every fitted component
    E_min=70, E_floor=25,
    bal_window=67,     # BAL: masses renormalised with this horizon (steps)
    # mixture of LoRA experts with a learned token router
    router_topk=2, aux_coef=0.01,
    # data and evaluation
    train_bytes=3_000_000, test_bytes=300_000, eval_windows=32, lambada_n=500, wikitext_windows=32,
    seeds_all=[0], seeds_pair=[1],
    # timing probe on the main study's model and adapter layout
    probe_model="EleutherAI/pythia-160m-deduped", probe_comp_layers=4, probe_K=12, probe_steps=60,
)
if OFFLINE:
    CFG.update(seq_len=128, batch=8, steps_per_domain=60, prefix=32, shared_rank=8, comp_rank=4, comp_layers=1, K_max=6,
               A_p=40, A_s=40, E_min=20, E_floor=8, bal_window=20, train_bytes=300_000, test_bytes=40_000, eval_windows=8,
               probe_comp_layers=2, probe_K=6, probe_steps=6, warmup=10)
if os.environ.get("UWFR_STEPS"): CFG["steps_per_domain"] = int(os.environ["UWFR_STEPS"])
AMP = CFG["amp"] and DEVICE == "cuda"
def autocast(): return torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=torch.float16, enabled=AMP)

CONDITIONS = {
    "LoRA-match":    dict(kind="lora"),                  # one LoRA on all layers, same trainable parameters as U at K_max
    "BAL":           dict(kind="pop", balanced=True),    # same model and algorithm, normalised masses, fully plastic
    "U":             dict(kind="pop"),                   # unbalanced WFR (the method)
    "MoLoRA-router": dict(kind="moe"),                   # same components, learned token-level top-2 router
}
print("conditions:", list(CONDITIONS))

## 2. Data: development stream
arXiv abstracts by primary category (streamed from `gfissore/arxiv-abstracts-2021`) and AG News sports. About 11% of each domain is held out, split by document. Documents are cached in Drive.

In [ ]:
# Development stream only: these domains are never used in the main study.
DEV = {
    "Number theory":       ("arxiv", ("math.NT",)),
    "Strongly corr. el.":  ("arxiv", ("cond-mat.str-el",)),
    "Computer vision":     ("arxiv", ("cs.CV",)),
    "Quant. finance":      ("arxiv", ("q-fin.",)),
    "Sports news":         ("agnews", "Sports"),
}
if OFFLINE:  # stand-ins for the code check
    DEV = {"World news": ("agnews", "World"), "Sports news": ("agnews", "Sports"), "Business news": ("agnews", "Business"),
           "Sci/Tech news": ("agnews", "Sci/Tech"), "Python code": ("local", "py")}
DOMAINS = list(DEV)
AG_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/{}.csv"
AG_NAMES = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

def _clean(s):
    s = re.sub(r"#(\d+);", lambda m: chr(int(m.group(1))), str(s))
    return re.sub(r"\s+", " ", html.unescape(s).replace("\\", " ")).strip()

def primary_category(cats):
    if isinstance(cats, (list, tuple)): cats = " ".join(map(str, cats))
    t = str(cats).split(); return t[0] if t else ""

def need(): return int((CFG["train_bytes"] + CFG["test_bytes"]) * 1.1)

def arxiv_from_hf(specs):
    from datasets import load_dataset
    ds = load_dataset("gfissore/arxiv-abstracts-2021", split="train", streaming=True).shuffle(seed=0, buffer_size=20_000)
    docs = {t: [] for t in specs}; size = {t: 0 for t in specs}; t0 = time.time()
    for n, r in enumerate(ds):
        pc = primary_category(r.get("categories", ""))
        for t, pref in specs.items():
            if size[t] < need() and pc.startswith(pref):
                d = _clean(r.get("title", "")) + ". " + _clean(r.get("abstract", "")); docs[t].append(d); size[t] += len(d.encode()) + 1
                break
        if all(size[t] >= need() for t in specs) or n > 2_000_000: break
        if n % 100_000 == 0 and n: print(f"  scanned {n:,} rows ({time.time() - t0:.0f}s): " + ", ".join(f"{t} {size[t] / 1e6:.1f}MB" for t in specs))
    return docs

def agnews_docs(classes):
    docs = {c: [] for c in classes}
    for split in ("train", "test"):
        path = os.path.join(ROOT, f"ag_news_{split}.csv")
        if not os.path.exists(path): urllib.request.urlretrieve(AG_URL.format(split), path)
        with open(path, newline="", encoding="utf-8") as f:
            for label, title, desc in csv.reader(f):
                if AG_NAMES[int(label)] in docs: docs[AG_NAMES[int(label)]].append(_clean(title) + ". " + _clean(desc))
    return docs

def local_docs():
    fs = sorted(glob.glob(os.path.dirname(os.__file__) + "/**/*.py", recursive=True)); random.Random(0).shuffle(fs); out, size = [], 0
    for f in fs:
        txt = open(f, encoding="utf-8", errors="ignore").read()
        for i in range(0, len(txt), 2000): out.append(txt[i:i + 2000]); size += 2000
        if size > need(): break
    return out

def load_docs():
    cache = os.path.join(ROOT, "docs_dev.json")
    if os.path.exists(cache):
        d = json.load(open(cache))
        if all(t in d for t in DOMAINS): print("loaded cached documents"); return d
    docs = {}
    ax = {t: v[1] for t, v in DEV.items() if v[0] == "arxiv"}
    if ax:
        print("downloading arXiv abstracts (streaming; about 5-10 minutes) ...")
        got = arxiv_from_hf({t: p[0] for t, p in ax.items()})
        for t in ax:
            if len(got[t]) < 200: raise RuntimeError(f"too few documents for {t}; check the connection and rerun")
        docs.update(got)
    ag = [v[1] for v in DEV.values() if v[0] == "agnews"]
    if ag:
        a = agnews_docs(ag)
        for t, v in DEV.items():
            if v[0] == "agnews": docs[t] = a[v[1]]
    for t, v in DEV.items():
        if v[0] == "local": docs[t] = local_docs()
    if not OFFLINE: json.dump(docs, open(cache, "w"))
    return docs

# ---- tokenizer ----
class ByteTok:                                    # offline stand-in only
    eos_token_id = 256; vocab_size = 257
    def encode(self, s): return list(s.encode("utf-8", "ignore"))
    def __call__(self, texts): return {"input_ids": [self.encode(t) for t in texts]}
if OFFLINE: TOK = ByteTok()
else:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(CFG["model"])

def encode_docs(docs, cap):
    ids, total = [], 0
    for i in range(0, len(docs), 512):
        for e in TOK(docs[i:i + 512])["input_ids"]:
            ids.extend(e); ids.append(TOK.eos_token_id); total += 1
        if len(ids) >= cap: break
    return torch.tensor(ids[:cap], dtype=torch.long)

def split_docs(d):
    d = list(d); random.Random(0).shuffle(d); test, size, i = [], 0, 0
    while i < len(d) and size < CFG["test_bytes"]: test.append(d[i]); size += len(d[i].encode()) + 1; i += 1
    train, size = [], 0
    while i < len(d) and size < CFG["train_bytes"]: train.append(d[i]); size += len(d[i].encode()) + 1; i += 1
    return encode_docs(train, 10 ** 9), encode_docs(test, 10 ** 9)

DOCS = load_docs(); TRAIN, TEST = {}, {}
for t in DOMAINS:
    TRAIN[t], TEST[t] = split_docs(DOCS[t])
    steps_tokens = CFG["steps_per_domain"] * CFG["batch"] * CFG["seq_len"]
    print(f"{t:20s} {len(DOCS[t]):>7,} docs | train {len(TRAIN[t]):>9,} tokens ({steps_tokens / max(1, len(TRAIN[t])):.1f} passes) | test {len(TEST[t]):>8,} tokens")

def domain_order(seed): return list(DOMAINS) if seed == 0 else random.Random(1000 + seed).sample(DOMAINS, len(DOMAINS))

def get_batch(data, B, T, gen):
    ix = torch.randint(len(data) - T - 1, (B,), generator=gen)
    x = torch.stack([data[i:i + T] for i in ix]); y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

def eval_windows(data, n, T):
    pos = np.linspace(0, len(data) - T - 2, n).astype(int)
    x = torch.stack([data[i:i + T] for i in pos]); y = torch.stack([data[i + 1:i + T + 1] for i in pos])
    return x.to(DEVICE), y.to(DEVICE)
EVAL = {t: eval_windows(TEST[t], CFG["eval_windows"], CFG["seq_len"]) for t in DOMAINS}

## 3. Model with shared and component adapters
The pretrained weights are frozen. The shared LoRA (rank 32) sits on every linear layer. Components (rank 16) sit on the attention output and MLP of the top 2 blocks. Each sequence uses one component, chosen by `CTX.gate`. LoRA-match's rank is computed so its trainable parameters equal U's at 8 components.

In [ ]:
class _Ctx:                                       # per-forward routing information shared by all adapted layers
    gate = None     # [B, K] one-hot (or soft) component weights per sequence
    aux = []        # router balance terms (MoLoRA)
CTX = _Ctx()

class AdaptedLinear(nn.Module):
    """Frozen base linear + shared LoRA + K component LoRAs (stacked on a leading K dim).
    Component k contributes ((x A_k) * g_k) B_k, with g from CTX.gate (sequence-level) or from a token router."""
    def __init__(s, base, shared_r, K, comp_r, router_topk=0):
        super().__init__(); s.base = base; fin, fout = base.in_features, base.out_features; s.K, s.topk = K, router_topk
        s.sA = nn.Parameter(torch.randn(fin, shared_r) / math.sqrt(fin)) if shared_r else None
        s.sB = nn.Parameter(torch.zeros(shared_r, fout)) if shared_r else None
        if K:
            s.cA = nn.Parameter(torch.randn(K, fin, comp_r) / math.sqrt(fin)); s.cB = nn.Parameter(torch.zeros(K, comp_r, fout))
        s.router = nn.Linear(fin, K) if (K and router_topk) else None
    def forward(s, x):
        y = s.base(x)
        if s.sA is not None: y = y + (x @ s.sA.to(x.dtype)) @ s.sB.to(x.dtype)
        if s.K:
            if s.router is not None:
                p = s.router(x).float().softmax(-1); top = p.topk(s.topk, -1)
                g = torch.zeros_like(p).scatter(-1, top.indices, top.values / top.values.sum(-1, keepdim=True))
                frac = F.one_hot(top.indices[..., 0], s.K).float().mean((0, 1)); CTX.aux.append(s.K * (frac * p.mean((0, 1))).sum())
            else:
                g = CTX.gate[:, None, :]
            h = torch.einsum("bti,kir->btkr", x, s.cA.to(x.dtype)) * g.to(x.dtype)[..., None]
            y = y + torch.einsum("btkr,kro->bto", h, s.cB.to(x.dtype))
        return y

def load_base(name):
    if OFFLINE:
        from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
        m = GPTNeoXForCausalLM(GPTNeoXConfig(vocab_size=257, hidden_size=96, num_hidden_layers=3, num_attention_heads=4,
                                             intermediate_size=384, max_position_embeddings=512))
        torch.manual_seed(1234)   # identical "pretrained" weights in every condition
        for p in m.parameters(): nn.init.normal_(p, 0, 0.02) if p.dim() > 1 else None
    else:
        from transformers import AutoModelForCausalLM
        try: m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
        except TypeError: m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)
    m.config.use_cache = False
    for p in m.parameters(): p.requires_grad_(False)
    return m.to(DEVICE)

class AdaptedLM(nn.Module):
    """Pythia (GPT-NeoX) with adapters. shared_r on every linear layer of every block; K components (comp_r) on the
    attention output and the MLP of the top comp_layers blocks."""
    def __init__(s, name, shared_r, K=0, comp_r=0, comp_layers=0, router_topk=0):
        super().__init__(); s.lm = load_base(name); s.K = K; layers = s.lm.gpt_neox.layers; L = len(layers)
        for i, blk in enumerate(layers):
            comp = K if i >= L - comp_layers else 0
            for parent, attr in ((blk.attention, "query_key_value"), (blk.attention, "dense"), (blk.mlp, "dense_h_to_4h"), (blk.mlp, "dense_4h_to_h")):
                k = 0 if attr == "query_key_value" else comp
                setattr(parent, attr, AdaptedLinear(getattr(parent, attr), shared_r, k, comp_r, router_topk if k else 0).to(DEVICE))
        s.V = s.lm.config.vocab_size
    def adapters(s): return [m for m in s.modules() if isinstance(m, AdaptedLinear)]
    def expert_params(s): return [p for a in s.adapters() if a.K for p in (a.cA, a.cB)]
    def trunk_params(s):
        out = []
        for a in s.adapters():
            if a.sA is not None: out += [a.sA, a.sB]
            if a.router is not None: out += list(a.router.parameters())
        return out
    def n_trainable(s): return sum(p.numel() for p in s.expert_params() + s.trunk_params())
    def seq_nll(s, x, y):
        """mean next-token NLL (nats) of every sequence"""
        logits = s.lm(input_ids=x).logits
        return F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), y.reshape(-1), reduction="none").view(y.shape).mean(1)
    @torch.no_grad()
    def clone(s, src, dst):
        for p in s.expert_params(): p[dst] = p[src]

def matched_rank(name, shared_r, K, comp_r, comp_layers):
    """rank of a single all-layer LoRA with the same number of trainable parameters as U at K components"""
    m = AdaptedLM(name, shared_r, K, comp_r, comp_layers); total = m.n_trainable()
    per_rank = sum(a.base.in_features + a.base.out_features for a in m.adapters())
    del m; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return max(1, round(total / per_rank)), total
MATCH_RANK, U_PARAMS = matched_rank(CFG["model"], CFG["shared_rank"], CFG["K_max"], CFG["comp_rank"], CFG["comp_layers"])
print(f"U at {CFG['K_max']} components: {U_PARAMS / 1e6:.2f}M trainable parameters -> LoRA-match rank {MATCH_RANK}")

## 4. Optimiser
`ParticleAdamW`: one step factor per component and one for the shared adapter. A factor of 0 leaves a component exactly unchanged.

In [ ]:
class ParticleAdamW:
    """Adam with one step factor per component (leading dim of the component tensors) and one for shared parameters.
    A factor of 0 leaves a component exactly unchanged (no momentum drift)."""
    def __init__(s, model, betas=(0.9, 0.999), eps=1e-8):
        s.ep, s.tp, s.b1, s.b2, s.eps = model.expert_params(), model.trunk_params(), betas[0], betas[1], eps
        s.m = {id(p): torch.zeros_like(p) for p in s.ep + s.tp}; s.v = {id(p): torch.zeros_like(p) for p in s.ep + s.tp}
        s.t_rows = torch.zeros(max(1, model.K), device=DEVICE); s.t_trunk = 0
    def zero_grad(s):
        for p in s.ep + s.tp: p.grad = None
    def reset(s, k):
        for p in s.ep: s.m[id(p)][k] = 0; s.v[id(p)][k] = 0
        s.t_rows[k] = 0
    @torch.no_grad()
    def step(s, lr, fac=None, fac_s=1.0):
        if fac is not None and s.ep:
            act = fac > 0
            if act.any():
                s.t_rows[act] += 1; bc1, bc2 = 1 - s.b1 ** s.t_rows[act], 1 - s.b2 ** s.t_rows[act]
                for p in s.ep:
                    if p.grad is None: continue
                    g = p.grad[act]; m, v = s.m[id(p)], s.v[id(p)]
                    m[act] = s.b1 * m[act] + (1 - s.b1) * g; v[act] = s.b2 * v[act] + (1 - s.b2) * g * g
                    sh = (-1,) + (1,) * (p.dim() - 1)
                    p[act] -= lr * fac[act].view(sh) * (m[act] / bc1.view(sh)) / ((v[act] / bc2.view(sh)).sqrt() + s.eps)
        if fac_s > 0 and s.tp:
            s.t_trunk += 1; bc1, bc2 = 1 - s.b1 ** s.t_trunk, 1 - s.b2 ** s.t_trunk
            for p in s.tp:
                if p.grad is None: continue
                m, v = s.m[id(p)], s.v[id(p)]
                m.mul_(s.b1).add_(p.grad, alpha=1 - s.b1); v.mul_(s.b2).addcmul_(p.grad, p.grad, value=1 - s.b2)
                p.add_((m / bc1) / ((v / bc2).sqrt() + s.eps), alpha=-lr * fac_s)

class LossScaler:
    def __init__(s, enabled, init=2.0 ** 14, interval=1000):
        s.enabled, s.scale, s.interval, s.good = enabled, (init if enabled else 1.0), interval, 0
    def unscale_and_check(s, params):
        if not s.enabled: return True
        inv, finite = 1.0 / s.scale, True
        for p in params:
            if p.grad is not None:
                p.grad.mul_(inv)
                if finite and not torch.isfinite(p.grad).all(): finite = False
        if not finite: s.scale /= 2; s.good = 0; return False
        s.good += 1
        if s.good % s.interval == 0: s.scale *= 2
        return True

## 5. Learners
`Population` implements U and BAL. Per batch:
1. **E-step:** each component's likelihood on the first 128 tokens; γ_c ∝ m_c ρ_c(x).
2. **Creation:** a component covers x if its loss is within δ of its running level. More than 30% of the batch uncovered by every fitted component = a new region, which goes to the newborn (cloned from the closest component).
3. **Reaction:** U adds the assigned count to each mass; BAL renormalises its masses. Both count raw evidence for promotion.
4. **Transport:** U uses min(1, A·b_c/m_c) for components and min(1, A·b/N_fit) for the shared adapter; BAL uses 1 for everything active.

In [ ]:
class LoRAMatch:
    """Single LoRA on all layers with the same trainable-parameter budget as U at K_max (plain sequential training)."""
    particles = False
    def __init__(s, cfg, name=None):
        s.cfg = cfg; s.model = AdaptedLM(name or cfg["model"], cfg.get("match_rank", MATCH_RANK)); s.opt = ParticleAdamW(s.model)
        s.scaler = LossScaler(AMP); s.it, s.log, s.skipped = 0, [], 0
    def _update(s, loss, lr, fac=None, fac_s=1.0):
        params = s.opt.ep + s.opt.tp
        s.opt.zero_grad(); (loss * s.scaler.scale).backward()
        if not s.scaler.unscale_and_check(params): s.skipped += 1; return False
        torch.nn.utils.clip_grad_norm_(params, s.cfg["clip"]); s.opt.step(lr, fac, fac_s); return True
    def step(s, x, y, lr):
        s.it += 1
        with autocast(): loss = s.model.seq_nll(x, y).mean()
        s._update(loss, lr); return float(loss.detach())
    @torch.no_grad()
    def seq_losses(s, x, y):
        with autocast(): return s.model.seq_nll(x, y).float()

class MoLoRARouter(LoRAMatch):
    """Same shared adapter and K components as U, all present from the start, mixed per token by a learned top-2 router."""
    def __init__(s, cfg, name=None):
        s.cfg = cfg; s.model = AdaptedLM(name or cfg["model"], cfg["shared_rank"], cfg["K_max"], cfg["comp_rank"], cfg["comp_layers"], cfg["router_topk"])
        s.opt = ParticleAdamW(s.model); s.scaler = LossScaler(AMP); s.it, s.log, s.skipped = 0, [], 0
        s.ones = torch.ones(cfg["K_max"], device=DEVICE)
    def step(s, x, y, lr):
        s.it += 1; CTX.aux = []
        with autocast():
            loss = s.model.seq_nll(x, y).mean(); total = loss + s.cfg["aux_coef"] * sum(CTX.aux) / max(1, len(CTX.aux))
        s._update(total, lr, s.ones, 1.0); return float(loss.detach())
    @torch.no_grad()
    def seq_losses(s, x, y):
        CTX.aux = []
        with autocast(): return s.model.seq_nll(x, y).float()

class Population(LoRAMatch):
    """Unbalanced WFR flow on a population of components (U); balanced=True gives the normalisation-only control (BAL).
       E-step    gamma_c(x) ∝ m_c rho_c(x) on the first `prefix` tokens (exact Bayes)
       Reaction  m_c += sequences assigned (U: growth only; BAL: renormalised)
       Transport component: min(1, A_p b_c / m_c); shared adapter: min(1, A_s b / N_fitted); newborn: free (BAL: all 1)
       Creation  a region (> nu of the batch) uncovered by every fitted component goes to the newborn"""
    particles = True
    def __init__(s, cfg, balanced=False, name=None, comp_layers=None, K=None):
        s.cfg, s.balanced = cfg, balanced; K = K or cfg["K_max"]; s.K = K
        s.model = AdaptedLM(name or cfg["model"], cfg["shared_rank"], K, cfg["comp_rank"], comp_layers or cfg["comp_layers"])
        s.opt = ParticleAdamW(s.model); s.scaler = LossScaler(AMP); s.it, s.log, s.skipped = 0, [], 0
        s.m = torch.zeros(K, device=DEVICE); s.is_ex = torch.zeros(K, dtype=torch.bool, device=DEVICE)
        s.promoted = torch.zeros(K, dtype=torch.bool, device=DEVICE); s.level = torch.full((K,), float("nan"), device=DEVICE)
        s.fresh = torch.zeros(K, dtype=torch.bool, device=DEVICE)
        s.ev = torch.zeros(K, device=DEVICE)   # cumulative evidence (sequences assigned); used for promotion in U and BAL alike
        s.free = 0; s.is_ex[0] = True; s.m[0] = 1e-3; s.fresh[0] = True; s.trace = []
    def exist(s): return s.is_ex.nonzero().squeeze(1)
    def mass(s): w = s.m * s.is_ex; return w / w.sum()
    def _spawn(s, parent):
        free = (~s.is_ex).nonzero().squeeze(1)
        if len(free) == 0: s.free = -1; return
        k = int(free[0]); s.model.clone(parent, k); s.opt.reset(k)
        s.is_ex[k] = True; s.m[k] = 1e-3; s.ev[k] = 0; s.level[k] = float("nan"); s.fresh[k] = True; s.free = k; s.log.append([s.it, "birth", k, int(parent)])
    def _promote(s, why):
        s.promoted[s.free] = True; s.log.append([s.it, "promote", s.free, why]); s._spawn(s.free)
    @torch.no_grad()
    def nll_matrix(s, x, y, parts, P=None):
        """[len(parts), B] mean NLL (nats/token) of each sequence under each component, on its first P tokens"""
        P = P or x.shape[1]; xp, yp = x[:, :P], y[:, :P]; B = x.shape[0]; out = []
        chunk = max(1, (CFG["batch"] * CFG["seq_len"]) // (B * P))
        for i in range(0, len(parts), chunk):
            pc = parts[i:i + chunk]; n = len(pc)
            CTX.gate = F.one_hot(torch.tensor(pc, device=DEVICE).repeat_interleave(B), s.K).float()
            with autocast(): out.append(s.model.seq_nll(xp.repeat(n, 1), yp.repeat(n, 1)).float().view(n, B))
        return torch.cat(out)
    def step(s, x, y, lr):
        s.it += 1; cfg = s.cfg; b = x.shape[0]; P = cfg["prefix"]
        ex = s.exist(); nll = s.nll_matrix(x, y, ex.tolist(), P)
        def uncovered(ex, nll):
            lev = s.level[ex][:, None]
            cov = (nll <= lev + cfg["delta"]) & ~torch.isnan(lev) & s.promoted[ex][:, None]
            return ~cov.any(0)
        unc = uncovered(ex, nll); region = float(unc.float().mean()) > cfg["nu_region"]; nov = unc & region
        if region and s.free >= 0:                                      # ---- creation ----
            fi = int((ex == s.free).nonzero())
            fc = float((nll[fi, nov] <= s.level[s.free] + cfg["delta"]).float().mean()) if not torch.isnan(s.level[s.free]) else 0.0
            if s.ev[s.free] >= cfg["E_floor"] * b and fc < 0.5: s._promote("new region")
            if s.free >= 0 and s.ev[s.free] < b:
                parent = int(ex[nll[:, nov].mean(1).argmin()])
                if parent != s.free: s.model.clone(parent, s.free); s.opt.reset(s.free); s.log.append([s.it, "reborn", s.free, parent])
            old = {int(c): nll[i] for i, c in enumerate(ex) if int(c) != s.free}
            ex = s.exist(); new = [int(c) for c in ex if int(c) not in old]
            if new:
                rows = dict(zip(new, s.nll_matrix(x, y, new, P)))
                nll = torch.stack([old[int(c)] if int(c) in old else rows[int(c)] for c in ex])
        logit = s.m[ex].log()[:, None] - nll * P                        # ---- E-step ----
        assign = ex[torch.distributions.Categorical(logits=logit.T).sample()]
        if s.free >= 0 and nov.any():
            assign[nov] = s.free
            if s.fresh[s.free]: s.log.append([s.it, "create", s.free, int(nov.sum())]); s.fresh[s.free] = False
        cnt = torch.bincount(assign, minlength=s.K).float()             # ---- reaction ----
        s.m = (s.m * (1 - 1 / cfg["bal_window"]) * s.is_ex + cnt) if s.balanced else (s.m + cnt); s.ev += cnt
        for i, c in enumerate(ex.tolist()):                             # running level on each component's own data
            sel = assign == c
            if sel.sum() >= 2:
                q = float(torch.quantile(nll[i, sel], cfg["level_q"]))
                s.level[c] = q if torch.isnan(s.level[c]) else (1 - cfg["level_rate"]) * s.level[c] + cfg["level_rate"] * q
        CTX.gate = F.one_hot(assign, s.K).float()                       # ---- M-step (transport) ----
        with autocast(): loss = s.model.seq_nll(x, y).mean()
        if s.balanced: fac, fac_s = (cnt > 0).float(), 1.0
        else:
            fac = torch.clamp(cfg["A_p"] * cnt / s.m.clamp(min=1), max=1.0)
            n_fit = float(s.m[s.promoted].sum()); fac_s = 1.0 if n_fit == 0 else min(1.0, cfg["A_s"] * b / n_fit)
        if s.free >= 0 and cnt[s.free] > 0: fac[s.free] = 1.0
        s._update(loss, lr, fac, fac_s)
        if s.free >= 0 and s.ev[s.free] >= cfg["E_min"] * b: s._promote("evidence")
        if s.it % 25 == 0: s.trace.append([s.it, round(float(unc.float().mean()), 3), round(fac_s, 4), int(len(ex))])
        return float(loss.detach())
    @torch.no_grad()
    def seq_losses(s, x, y, return_post=False):
        """mixture over components: -log sum_c pi_c rho_c(x), per token"""
        ex = s.exist(); T = x.shape[1]; nll = s.nll_matrix(x, y, ex.tolist())
        lj = s.mass()[ex].clamp_min(1e-30).log()[:, None] - nll * T
        mix = -torch.logsumexp(lj, 0) / T
        return (mix, lj.softmax(0), ex) if return_post else mix

def make_learner(name, cfg=None, model_name=None):
    cfg = cfg or CFG; k = CONDITIONS[name]["kind"]
    if k == "lora": return LoRAMatch(cfg, model_name)
    if k == "moe": return MoLoRARouter(cfg, model_name)
    return Population(cfg, balanced=CONDITIONS[name].get("balanced", False), name=model_name)

## 6. Evaluation
Held-out loss per domain (nats per token). Population models are scored as the sequence-level mixture Σ π_c ρ_c(x) with π = m/Σm; domain labels are used only to compute the routing ARI. General ability: LAMBADA last-word accuracy (500 examples; populations use the component with the best context likelihood) and WikiText-103 test loss, at the start and the end.

In [ ]:
@torch.no_grad()
def evaluate_domains(L):
    """held-out loss (nats/token) per domain; for populations also the posterior over components (labels: analysis only)"""
    L.model.eval(); out, post, labels, routes = {}, {}, [], []
    for di, t in enumerate(DOMAINS):
        x, y = EVAL[t]; vals, P = [], []
        for i in range(0, len(x), 8):
            if L.particles:
                mix, pst, ex = L.seq_losses(x[i:i + 8], y[i:i + 8], True); vals.append(mix); P.append(pst)
                routes += ex[pst.argmax(0)].tolist(); labels += [di] * len(mix)
            else: vals.append(L.seq_losses(x[i:i + 8], y[i:i + 8]))
        out[t] = float(torch.cat(vals).mean())
        if L.particles: post[t] = torch.cat(P, 1).mean(1).tolist()
    L.model.train()
    res = dict(loss=out)
    if L.particles:
        from sklearn.metrics import adjusted_rand_score
        res.update(post=post, ex=L.exist().tolist(), mass=L.mass().tolist(), ari=float(adjusted_rand_score(labels, routes)))
    return res

GENERAL = None
def load_general():
    """LAMBADA (last-word accuracy) and WikiText-103 test windows: general ability retained from pretraining"""
    global GENERAL
    if GENERAL is not None or OFFLINE: return GENERAL
    from datasets import load_dataset
    lam = load_dataset("EleutherAI/lambada_openai", "default", split="test")["text"][:CFG["lambada_n"]]
    ex = []
    for t in lam:
        ctx, w = t.rsplit(" ", 1); c = TOK(ctx)["input_ids"]; g = TOK(" " + w)["input_ids"]; ex.append((c, g))
    wt = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")["text"]
    ids = torch.tensor(TOK(["".join(wt[:4000])])["input_ids"][0], dtype=torch.long)
    GENERAL = dict(lambada=ex, wiki=eval_windows(ids, CFG["wikitext_windows"], CFG["seq_len"]))
    return GENERAL

@torch.no_grad()
def evaluate_general(L):
    G = load_general()
    if G is None: return {}
    L.model.eval()
    xw, yw = G["wiki"]; wl = torch.cat([L.seq_losses(xw[i:i + 8], yw[i:i + 8]) for i in range(0, len(xw), 8)]).mean()
    parts = L.exist().tolist() if L.particles else [None]; correct = 0
    for i in range(0, len(G["lambada"]), 16):
        batch = G["lambada"][i:i + 16]; n = max(len(c) + len(g) for c, g in batch)
        X = torch.full((len(batch), n), TOK.eos_token_id, dtype=torch.long)
        for j, (c, g) in enumerate(batch): X[j, :len(c) + len(g)] = torch.tensor(c + g)
        X = X.to(DEVICE); best_ctx = torch.full((len(batch),), float("inf")); ok_best = torch.zeros(len(batch), dtype=torch.bool)
        for p in parts:
            if p is not None: CTX.gate = F.one_hot(torch.full((len(batch),), p, device=DEVICE), L.K).float()
            CTX.aux = []
            with autocast(): logits = L.model.lm(input_ids=X).logits.float()
            lp = logits.log_softmax(-1)                       # stays on the GPU; only a few values are copied back
            for j, (c, g) in enumerate(batch):
                ctxnll = float(-lp[j, torch.arange(len(c) - 1, device=DEVICE), torch.tensor(c[1:], device=DEVICE)].sum())
                ok = bool((lp[j, len(c) - 1:len(c) + len(g) - 1].argmax(-1).cpu() == torch.tensor(g)).all())
                if ctxnll < best_ctx[j]: best_ctx[j] = ctxnll; ok_best[j] = ok
        correct += int(ok_best.sum())
    L.model.train()
    return dict(wikitext_loss=float(wl), lambada_acc=correct / len(G["lambada"]))

def summarize(M):
    M = np.array(M); T = M.shape[1]
    return dict(final=float(M[T].mean()), forgetting=float(np.mean([M[T, t] - M[t + 1, t] for t in range(T - 1)])),
                plasticity=float(np.mean([M[t + 1, t] for t in range(T)])),
                forgot_share=float(np.mean([(M[T, t] - M[t + 1, t]) / max(1e-6, M[0, t] - M[t + 1, t]) for t in range(T - 1)])))

## 7. Training loop and storage

In [ ]:
RESULTS_PATH = os.path.join(ROOT, "results.json")
def load_results(): return json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
CFG_KEY = {k: v for k, v in CFG.items() if k not in ("seeds_all", "seeds_pair")}

def train_one(name, seed):
    order = domain_order(seed); T, S = len(order), CFG["steps_per_domain"]
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    gen = torch.Generator().manual_seed(seed); L = make_learner(name)
    print(f"  trainable parameters: {L.model.n_trainable() / 1e6:.2f}M")
    ev = evaluate_domains(L); M = np.zeros((T + 1, T)); M[0] = [ev["loss"][t] for t in order]
    gen0 = evaluate_general(L); stages, losses, hashes, step, t0 = [], [], [], 0, time.time()
    for s, dom in enumerate(order):
        for it in range(S):
            lr = CFG["lr"] * min(1.0, (step + 1) / CFG["warmup"])
            x, y = get_batch(TRAIN[dom], CFG["batch"], CFG["seq_len"], gen)       # the learner only ever sees (x, y)
            if it == 0: hashes.append(hashlib.md5(x.cpu().numpy().tobytes()).hexdigest()[:12])
            losses.append(L.step(x, y, lr)); step += 1
            if step == 50:
                per = (time.time() - t0) / step; print(f"  [{name} s{seed}] {per * 1000:.0f} ms/step -> about {per * T * S / 60:.0f} min of training")
        ev = evaluate_domains(L); M[s + 1] = [ev["loss"][t] for t in order]; stages.append(ev)
        extra = f" | parts {len(ev['ex'])}, mass " + " ".join(f"{ev['mass'][c]:.2f}" for c in ev["ex"]) + f", ARI {ev['ari']:.2f}" if L.particles else ""
        print(f"  [{name} s{seed}] after {dom:20s}: " + " ".join(f"{M[s + 1, i]:.3f}" for i in range(T)) + f"  ({(time.time() - t0) / 60:.1f} min){extra}")
    gen1 = evaluate_general(L)
    res = dict(M=M.tolist(), order=order, minutes=(time.time() - t0) / 60, skipped=L.skipped, steps=step, losses=losses,
               batch_hashes=hashes, stages=stages, events=L.log, trace=getattr(L, "trace", []), general0=gen0, general1=gen1,
               trainable=L.model.n_trainable(), **summarize(M))
    return res

def run_condition(name, seeds):
    R = load_results()
    if "_cfg" in R and R["_cfg"] != json.loads(json.dumps(CFG_KEY)):
        raise RuntimeError("results.json was made with different settings; use a new folder (change ROOT) for new settings.")
    for seed in seeds:
        if str(seed) in R.get(name, {}): print(f"{name} seed {seed}: already done, skipping"); continue
        print(f"=== {name} | seed {seed} | order: {' -> '.join(domain_order(seed))} ===")
        res = train_one(name, seed)
        R = load_results(); R["_cfg"] = CFG_KEY; R.setdefault(name, {})[str(seed)] = res; json.dump(R, open(RESULTS_PATH, "w"))
        g = f" | LAMBADA {res['general0'].get('lambada_acc', float('nan')):.3f}->{res['general1'].get('lambada_acc', float('nan')):.3f}" if res["general1"] else ""
        print(f"  final {res['final']:.4f} | forgetting {res['forgetting']:+.4f} | plasticity {res['plasticity']:.4f} nats/token{g}")
        if DEVICE == "cuda": torch.cuda.empty_cache()
print("ready. saved so far:", {k: sorted(v) for k, v in load_results().items() if k != "_cfg"})

## 8. Runs
Seed 0 for all four conditions, then seed 1 for U and LoRA-match to see the seed-to-seed spread.

#### LoRA-match, seed 0

In [ ]:
run_condition("LoRA-match", CFG["seeds_all"])

#### BAL, seed 0

In [ ]:
run_condition("BAL", CFG["seeds_all"])

#### U, seed 0

In [ ]:
run_condition("U", CFG["seeds_all"])

#### MoLoRA-router, seed 0

In [ ]:
run_condition("MoLoRA-router", CFG["seeds_all"])

#### U and LoRA-match, seed 1

In [ ]:
run_condition("U", CFG["seeds_pair"])
run_condition("LoRA-match", CFG["seeds_pair"])

## 9. Timing probe for the main study
60 steps each on **Pythia-160M** with the main study's layout (components on the top 4 blocks, all 12 present: the worst case for U). The projection uses the plan's run counts: 13 segments × 300 steps, +15% for evaluation, +3 h for Tier 0.

In [ ]:
def timing_probe():
    """seconds per step on the main study's model and adapter layout (worst case: all components present)"""
    path = os.path.join(ROOT, "timing.json")
    if os.path.exists(path): T = json.load(open(path)); print("timing (cached):", T); return T
    name = CFG["model"] if OFFLINE else CFG["probe_model"]; B, Tn = CFG["batch"], CFG["seq_len"]
    gen = torch.Generator().manual_seed(0); dom = DOMAINS[0]
    mr, _ = matched_rank(name, CFG["shared_rank"], CFG["probe_K"], CFG["comp_rank"], CFG["probe_comp_layers"])
    pcfg = dict(CFG, K_max=CFG["probe_K"], comp_layers=CFG["probe_comp_layers"], match_rank=mr)
    out = {}
    def clock(L, label):
        for i in range(CFG["probe_steps"] + 5):
            if i == 5:
                if DEVICE == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
                t0 = time.time()
            x, y = get_batch(TRAIN[dom], B, Tn, gen); L.step(x, y, CFG["lr"])
        if DEVICE == "cuda": torch.cuda.synchronize()
        out[label] = dict(s_per_step=(time.time() - t0) / CFG["probe_steps"],
                          peak_gb=(torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0))
        print(f"  {label:14s} {out[label]['s_per_step']:.3f} s/step, peak {out[label]['peak_gb']:.1f} GB"); del L
        if DEVICE == "cuda": torch.cuda.empty_cache()
    L = Population(pcfg, name=name, comp_layers=CFG["probe_comp_layers"], K=CFG["probe_K"])     # all components present
    L.is_ex[:] = True; L.promoted[:] = True; L.m[:] = 1000.0; L.level[:] = 10.0; L.free = -1; clock(L, "U")
    clock(LoRAMatch(pcfg, name), "LoRA-match"); clock(MoLoRARouter(pcfg, name), "MoLoRA-router")
    class FullFT(LoRAMatch):
        def __init__(s):
            s.cfg = pcfg; s.model = AdaptedLM(name, 0); s.it, s.log, s.skipped = 0, [], 0
            for p in s.model.lm.parameters(): p.requires_grad_(True)
            s.o = torch.optim.AdamW(s.model.lm.parameters(), lr=5e-5); s.sc = torch.amp.GradScaler(enabled=AMP)
        def step(s, x, y, lr):
            with autocast(): loss = s.model.seq_nll(x, y).mean()
            s.o.zero_grad(); s.sc.scale(loss).backward(); s.sc.step(s.o); s.sc.update(); return float(loss.detach())
    clock(FullFT(), "Full-FT")
    json.dump(out, open(path, "w")); return out

def project_main(T):
    """GPU-hours of the main study (13 segments x 300 steps, plan section 'Compute budget'), +15% for evaluation, +3 h Tier 0"""
    steps = 13 * 300
    runs = {"U": 17, "MoLoRA-router": 3, "LoRA-match": 21, "Full-FT": 3}
    h = sum(n * steps * T[k]["s_per_step"] for k, n in runs.items()) * 1.15 / 3600 + 3
    return h

TIMING = timing_probe()
print(f"projected main study: {project_main(TIMING):.1f} GPU-hours")

## 10. Results and decision
The table, the loss matrices, the forgetting/plasticity plot, the routing heatmap, then the go/no-go table with the decision rule from the plan.

In [ ]:
def report():
    R = load_results(); names = [n for n in CONDITIONS if n in R]
    print(f"{'condition':15s} {'seed':>4s} {'final':>7s} {'forget':>7s} {'share':>6s} {'plast':>7s} {'LAMBADA 0->1':>13s} {'Wiki 0->1':>13s} {'min':>5s}")
    for n in names:
        for sd, r in sorted(R[n].items()):
            g0, g1 = r.get("general0") or {}, r.get("general1") or {}
            lam = f"{g0.get('lambada_acc', float('nan')):.3f}->{g1.get('lambada_acc', float('nan')):.3f}"
            wik = f"{g0.get('wikitext_loss', float('nan')):.3f}->{g1.get('wikitext_loss', float('nan')):.3f}"
            print(f"{n:15s} {sd:>4s} {r['final']:7.4f} {r['forgetting']:+7.4f} {r['forgot_share']:6.2f} {r['plasticity']:7.4f} {lam:>13s} {wik:>13s} {r['minutes']:5.1f}")
    for n in names:
        r = R[n]["0"] if "0" in R[n] else next(iter(R[n].values())); M = np.array(r["M"]); T = M.shape[1]
        print(f"\n{n}: loss matrix (rows = after segment, columns = domains in learning order {r['order']})")
        for i in range(T + 1): print(f"{('init' if i == 0 else 'seg ' + str(i)):>6s} " + " ".join(f"{M[i, t]:6.3f}" for t in range(T)))
    # plots: forgetting/plasticity, loss curves, routing
    fig, ax = plt.subplots(figsize=(6.5, 4.4))
    for n in names:
        for sd, r in R[n].items(): ax.scatter(r["plasticity"], r["forgetting"], s=60, label=f"{n} s{sd}")
    ax.set_xlabel("plasticity: loss right after learning (nats/token, lower = better)"); ax.set_ylabel("forgetting (nats/token, lower = better)")
    ax.grid(alpha=.3); ax.legend(fontsize=7); plt.tight_layout(); plt.show()
    fig, axes = plt.subplots(1, len(names), figsize=(4.3 * len(names), 3.3), squeeze=False)
    for ax, n in zip(axes[0], names):
        r = R[n]["0"]; M = np.array(r["M"])
        for t, d in enumerate(r["order"]): ax.plot(range(M.shape[0]), M[:, t], marker="o", label=d)
        ax.set_title(n); ax.set_xlabel("segment"); ax.set_ylabel("loss"); ax.grid(alpha=.3)
    axes[0][0].legend(fontsize=7); plt.tight_layout(); plt.show()
    pn = [n for n in names if CONDITIONS[n]["kind"] == "pop"]
    if pn:
        fig, axes = plt.subplots(1, len(pn), figsize=(5 * len(pn), 3.3), squeeze=False)
        for ax, n in zip(axes[0], pn):
            r = R[n]["0"]; st = r["stages"][-1]; Pm = np.array([st["post"][d] for d in r["order"]])
            im = ax.imshow(Pm, vmin=0, vmax=1, aspect="auto"); ax.set_yticks(range(len(r["order"]))); ax.set_yticklabels(r["order"], fontsize=8)
            ax.set_xticks(range(len(st["ex"]))); ax.set_xticklabels([f"{c}\n{st['mass'][c]:.2f}" for c in st["ex"]], fontsize=7)
            ax.set_title(f"{n}: posterior of each domain's text (ARI {st['ari']:.2f})", fontsize=9)
        fig.colorbar(im, ax=axes[0].tolist(), shrink=.8); plt.show()
        for n in pn: print(n, "events:", [e for e in R[n]["0"]["events"] if e[1] != "reborn"])

def creation_stats(r, window=50):
    S = CFG["steps_per_domain"]; T = len(r["order"]); cr = [e[0] for e in r["events"] if e[1] == "create"]
    hit = [any(sw * S < c <= sw * S + window for c in cr) for sw in range(1, T)]
    spurious = [c for c in cr if not any(sw * S < c <= sw * S + window for sw in range(0, T))]
    return sum(hit), T - 1, len(spurious)

def go_no_go():
    R = load_results(); tim = timing_probe(); rows = []
    def get(n, sd="0"): return R.get(n, {}).get(sd)
    lm, bal, u, mo = get("LoRA-match"), get("BAL"), get("U"), get("MoLoRA-router")
    if not (lm and bal and u): print("run LoRA-match, BAL and U (seed 0) first"); return
    rows.append(("G1", "forgetting exists", f"LoRA-match forgetting {lm['forgetting']:+.3f} nats, share {lm['forgot_share']:.0%}",
                 lm["forgetting"] >= 0.05 or lm["forgot_share"] >= 0.20))
    seeds = [sd for sd in R["U"] if sd in R.get("LoRA-match", {})]
    g2 = all(R["U"][sd]["forgetting"] <= 0.8 * R["LoRA-match"][sd]["forgetting"] for sd in seeds) and u["forgetting"] <= 0.7 * bal["forgetting"]
    rows.append(("G2", "U helps", f"U {u['forgetting']:+.3f} vs BAL {bal['forgetting']:+.3f} vs LoRA-match {lm['forgetting']:+.3f} (seeds {seeds})", g2))
    rows.append(("G3", "plasticity cost small", f"U {u['plasticity']:.3f} vs LoRA-match {lm['plasticity']:.3f}", u["plasticity"] - lm["plasticity"] <= 0.05))
    hit, n, spur = creation_stats(u); ari = u["stages"][-1]["ari"]
    rows.append(("G4", "mechanism works", f"creation at {hit}/{n} switches, {spur} spurious, ARI {ari:.2f}", hit >= 3 and spur <= 2 and ari >= 0.5))
    hours = project_main(tim)
    rows.append(("G5", "fits the budget", f"projected main study {hours:.1f} GPU-h", hours <= 27))
    nE = CFG["E_min"] - 1; same = np.max(np.abs(np.array(u["losses"][:nE]) - np.array(bal["losses"][:nE]))) if nE > 0 else 0.0
    hashes = all(R[n_]["0"]["batch_hashes"] == lm["batch_hashes"] for n_ in R if n_ != "_cfg" and "0" in R[n_])
    skipped = max(R[n_][sd]["skipped"] / R[n_][sd]["steps"] for n_ in R if n_ != "_cfg" for sd in R[n_])
    rows.append(("G6", "sanity", f"identical batches {hashes}; U vs BAL max loss diff before promotion {same:.1e}; max skipped {skipped:.1%}",
                 hashes and same < 5e-3 and skipped < 0.01))
    print(f"\n{'':4s} {'criterion':22s} {'pass':5s} evidence")
    for code, what, ev, ok in rows: print(f"{code:4s} {what:22s} {'YES' if ok else 'no':5s} {ev}")
    ok = {c: o for c, _, _, o in rows}
    if all(ok.values()): d = "ALL PASS: freeze the settings, pre-register, run the main study."
    elif not ok["G1"]: d = "G1 fails: stream too gentle; raise the share of strong-shift domains and rerun once."
    elif not ok["G2"] and ok["G4"]: d = "G2 fails with G4 passing: mechanism works but does not pay at this scale; do not run the main study as planned."
    elif not ok["G4"]: d = "G4 fails: retune delta and nu_region on this development stream only, rerun once."
    elif not ok["G5"]: d = "G5 fails: cut segments 13 -> 10 or steps 300 -> 250 and re-estimate."
    else: d = "Mixed: check G3/G6 details before deciding; record any change and its reason."
    print("\nDECISION:", d)
    if mo: print(f"(MoLoRA-router for reference: final {mo['final']:.3f}, forgetting {mo['forgetting']:+.3f})")

report()
go_no_go()

## Reading the decision
- **All pass:** freeze the settings, pre-register (OSF), run the main study.
- **G1 fails:** the stream is too gentle for a pretrained model (Pythia's training data already contains arXiv). Raise the share of strong-shift domains and rerun once.
- **G4 fails, others pass:** retune `delta` and `nu_region` on this development stream only, in a new folder, and rerun once.
- **G2 fails while G4 passes:** the mechanism works but does not pay at this scale; do not run the main study as planned.
- **G5 fails:** cut the main study from 13 to 10 segments or from 300 to 250 steps.

Record every change and its reason before filing the pre-registration.